# Prompt-Based Summarization with Mistral-7B

Loads a 4‑bit quantized Mistral‑7B model and uses two different prompts to summarize long essays, then prints generated summaries alongside source snippets and earlier T5‑based outputs for qualitative comparison.

In [ ]:
!pip install transformers[torch] datasets evaluate rouge_score pandas
!pip install accelerate bitsandbytes
!pip install bert_score
!pip install py7zr

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=24cc639d92d715983947e7c0e14ee10bbef02d4e7b09161c28a55392d8c03602
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.7/142.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 429.9/429.9 kB 45.1 MB/s eta 0:00:00


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

llm_model_name = "mistralai/Mistral-7B-Instruct-v0.2"
device = "cuda"

print("Loading 4-bit Mistral model...")
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    quantization_config=bnb_config,
    device_map=device
)
print("Mistral model loaded.")



Loading 4-bit Mistral model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Mistral model loaded.


In [ ]:
prompt_template_1 = """[INST]
Summarize the following essay into a concise paragraph. Focus on the main argument and key takeaways.
[/INST]
"""

prompt_template_2 = """[INST]
Act as an expert academic editor. Read the following essay and write a 3-sentence abstract that would appear in a scholarly journal.
[/INST]
"""
prompts = [prompt_template_1, prompt_template_2]


essay_for_part3_1 = """When I was 12, on family vacation in New Mexico, I watched a group of elaborately-costumed Navajo men belt out one intimidating song after the next. They executed a set of beautifully coordinated dance turns to honour the four cardinal directions, each one symbolising sacred gifts from the gods. Yet the tourist-packed audience lost interest and my family, too, prepared to leave. Then, all of a sudden, the dancers were surprised by a haunting, muscled old man adorned with strange pendants, animal skulls, and scars etching patterns into his body and face. Because the dancers were obviously terrified of this man I, too, became afraid and wanted to run, but we all stood rooted to the spot as he walked silently and majestically into the desert night. Afterwards, the lead dancer apologised profusely for the tribe’s shaman, or medicine man: he was holy but a bit eccentric. My 12-year-old self wondered how one might become like this extraordinary individual, so singular, respected and brave he could take the desert night alone. That question has fuelled much of my neuroscience through the years. As I studied the brain, I found that the right arrangement of neural circuitry and chemistry could generate astonishingly creative and holy persons on the one hand, or profoundly delusional, even violent, fanatics on the other. To intensify the ‘god effect’ in people already attracted to religious ideas, my studies revealed, all we had to do was boost the activity of the neurotransmitter, dopamine, crucial for balanced emotion and thought, on the right side of the brain. But should dopamine spike too high, murderous impulses like terrorism and jihad could rear up instead. Evidence that religion can produce extraordinary behaviours goes back to the dawn of human history, when our ancestors started burying the dead and produced remarkable, ritual art on cave walls. One of the first signs of religious consciousness dates to the upper Paleolithic, some 25,000 years ago, when a boy, also about age 12, crawled through hundreds of metres of pitch black, deep cave space, probably guided only by a flickering flame held in one hand and some fleetingly illuminated paintings on cave walls. When the boy reached a cul-de-sac in the bowels of the cave, he put red ochre onto his hand and made a print on the wall. Then he climbed out of the cul-de-sac and – we can surmise, given his skill and the fact that his bones have not been found – made it out alive. But where did this boy get his courage? And why leave a handprint on the wall of a remote cave deep in the bowels of the earth? Some experts in cave art think the boy was performing a religious obligation. He, like others who made similar treks into the caves, was leaving a votive offering to the spirit world or gods and becoming a holy man – much like the majestic and terrifying Indian man I had seen when I was 12. Dopamine probably fuelled his brain. Throughout the centuries, bountiful dopamine has given rise to gifted leaders and peacemakers (Gandhi, Martin Luther King, Catherine of Siena), innovators (Zoroaster), seers (the Buddha), warriors (Napoleon, Joan of Arc), teachers of whole civilisations (Confucius) and visionaries (Laozi). Some of them founded not only enduring religious traditions but also profoundly influenced the cultures and civilisations associated with those traditions. But dopamine-fuelled religion has also unleashed monsters: Jim Jones (the ‘minister’ who persuaded hundreds of his followers to commit suicide) and the cult Aum Shinrikyo, whose leader had his adherents release sarin gas on the subways of Japan. Think of the fanatic terrorists of al Qaeda, who gave their lives to attack New York’s twin towers and the Pentagon on 11 September 2001. The neurological line between the saint and the savage, the creative and the unconscionable, turns out to be razor-thin As 9/11 suggests, the neurological line between the saint and the savage, the creative and the unconscionable, turns out to be razor-thin. Just look at the bounty of evidence showing that families of extraordinarily creative individuals often include members with histories of insanity, sometimes even criminal insanity. Genes that produce brains capable of unusually creative associations or ideas are also more likely to produce (in the same individual or in members of his/her family) brains vulnerable to loose or bizarre associations. The medical literature abounds with descriptions of creative bursts following infusion of dopamine-enhancing drugs such as l-dopa (levodopa), used to treat Parkinson’s Disease (PD). Bipolar illness, which sends sufferers into prolonged bouts of dopamine-fuelled mania followed by devastating spells of depressive illness, can sometimes produce work of amazing virtuosity during the manic phase. Often these individuals refuse to take anti-dopamine drugs that can prevent the manic episodes precisely because they value the creative activity of which they are capable during these altered states. Hallucinogenic drugs such as Psilocybin and LSD, which indirectly stimulate dopamine activity in the brain’s frontal lobes, can produce religious experience even in the avowedly non-religious. These hallucinogens produce vivid imagery, sometimes along with near psychotic breaks or intense spiritual experience, all tied to stimulation of dopamine receptors on neurons in the limbic system, the seat of emotion located in the midbrain, and in the prefrontal cortex, the upper brain that is the center of complex thought. Given all these fascinating correlations, sometime after the attack on the twin towers in New York City, I began to hypothesize that dopamine might provide a simple explanation for the paradoxical god effect. When dopamine in the limbic and prefrontal regions of the brain was high, but not too high, it would produce the ability to entertain unusual ideas and associations, leading to heightened creativity, inspired leadership and profound religious experience. When dopamine was too high, however, it would produce mental illness in genetically vulnerable individuals. In those who had been religious before, fanaticism could be the result. While pursuing these ideas, I had a lucky break during routine office hours at the VA (Veterans Administration) Boston Healthcare System, where I regularly treat US veterans. I was doing a routine neuropsychological examination of a tall, distinguished elderly man with Parkinson’s Disease. This man was a decorated Second World War veteran and obviously intelligent. He had made his living as a consulting engineer but had slowly withdrawn from the working world as his symptoms progressed. His withdrawal was selective: he did not quit everything, his wife explained. ‘Just social parts of his work, some physical stuff and unfortunately his private religious devotions.’ When I asked what she meant by ‘devotions’ she replied that he used to pray and read his Bible all the time, but since the onset of the disease he had done so less and less. When I asked the patient himself about his religious interests, he replied that they seemed to have vanished. What was so striking was that he said he was quite unhappy about that fact. What appeared to be keeping him from his ‘devotions’ was that he found them ‘hard to fathom’. He had not stopped wanting to believe and practise his religion but simply found it more difficult to do so. This was a man whose intelligence was above average, who apparently had been religious all his life and who could easily answer questions about religious ideas and doctrines. It was not an intellectual deficit that was the problem. When I asked him directly whether he had now rejected religion as false he said: ‘By no means!’ The difficulty he had was accessing his religious memories, feelings and experiences, in particular. Other equally complex ideas were still easily available to him, but religion as a sphere of interest for this man was nearly impossible. The primary pathology associated with PD is a loss of dopamine activity, hypothesized for years to drive ‘hedonic reward’ or pleasure – that sense of well-being we all feel when we indulge in an experience like good food or sex. Whenever dopamine release occurred, proponents held, we would get a small hit of pleasure. That story made sense because many drugs of abuse, such as cocaine or amphetamine, stimulate dopamine activity in the midbrain. But recent research had revealed something more complex. A Cambridge University neuroscientist named Wolfram Schultz had shown that dopamine was not a simple pleasure molecule, delivering a simple reward. Instead, it alerted us only to unexpected rewards, spiking when the prize delivered far exceeded the expected result. Unexpected visions can define the most innovative artists, the most divergent philosophers and anyone who finds a sense of ecstasy in the beauty and strangeness of the world To tease out this nuance, Schultz used a simple experimental design: he delivered varying quantities of fruit juice to monkeys while simultaneously recording activity in the monkey’s midbrain, the seat of emotion, where dopamine neurons were dense. He found that the neurons fired most intensely not when the monkey got a juice reward but when that reward was unexpectedly large. In short, dopamine neurons were oriented towards the pickup of new and significant rewards, novelty of the highest order for the individual. Since Schultz’s pioneering work in the midbrain, others have mapped out similar signaling patterns when dopamine activity moves into the prefrontal lobes, which mediate the most complex of thinking and creative processes, unique to humans. But how would these new findings explain my PD patient’s difficulty in accessing religious ideas? Suppose religion created spectacular individuals because it pushed them into looking for unexpected rewards – a sense of transcendence or the pleasure of doing good – rather than all the usual rewards such as money or sex that the rest of us constantly pursue. Pursuit of unusual ideas could likewise be facilitated by dopamine, heightening creativity, too. Here, I thought, was where science and religion actually meet. Like the most creative scientists, the most consistently religious individuals would be motivated only by things that consistently triggered surging dopamine and the unexpected rewards circuits in the prefrontal lobes of the brain: awe, fear, reverence and wonder. Such unexpected visions would define the most innovative artists, the most divergent philosophers and anyone who could find a sense of ecstasy in the beauty and strangeness of the world. In the genetically vulnerable, it would take just a little more juice to activate homicidal fanatics like the terrorists of 9/11. Again, I tested these ideas on my PD patients. After screening for ‘religiousness’ through a questionnaire given to 71 vets in all, I found that a pattern was emerging. Of those with religious leanings prior to getting sick, only a subgroup lost religious fervor after illness set in. These were patients with ‘left-onset disease’ – meaning that their muscle problems had begun on the left side of the body, correlating with dysfunction in the right prefrontal regions of the brain. Those with left-onset disease reported significantly lower scores on all dimensions of religiosity (spiritual experiences, daily rituals, prayer and meditation) compared to those with right-onset disease. How could I explain these results? I surmised it was loss of dopamine in the right half of the brain. To test that hypothesis, my team devised a ‘priming’ experiment to see if PD patients could access religious concepts as easily as other, equally complex ideas. To conduct priming experiments, you typically ‘prime’ or briefly present, a person with a word semantically related to a second, target word. For instance, the word ‘rose’ can be used as a prime for ‘violet’. The target word, ‘violet’, will be more quickly recognized following a prime with ‘rose’ than it would be if the prime had been something unrelated, such as ‘stamp’. In our priming experiments with religious words we found that healthy volunteers were much quicker to recognize ‘worship God’ as a valid phrase when they were first presented with ‘pray quietly’ as compared with a control phrase. But this did not hold for PD patients with left-onset disease and damage on the right side of the brain! Relative to both right-onset PD patients and healthy volunteers, patients with left-onset disease did not benefit from subliminal presentation of the religious phrase, although their priming patterns for non-religious control phrases such as ‘pay taxes’ and ‘serve jury’ were normal in every way. These findings convinced me that my theory was at least partially correct: the dopamine receptors responsible for that transcendent, outsize sense of reward were dysfunctional on the right side of the brain. But I still had to rule out competing ideas. One long-standing theory, suggested most prominently by Freud, ascribes religious commitment to anxiety. Put simply, the theory says that religion with its promise of an afterlife quells the free-floating anxiety caused by fear of death. This was a problem for me, given that my unexpected rewards theory of religion predicts the exact opposite: rather than eschewing fear, the religious would seek it out as one of the most novel, exciting and intense emotions served up by the brain. I therefore attempted to pit the two theories against one another in another priming experiment. Over the course of several interview sessions, I told my PD patients a short story about a person climbing stairs in a hospital, ultimately coming across a surprise. Only the last sentence differed from one version to the next. In one ending the person witnessed a death; in a second he witnessed a religious ritual; and in the third, he saw a breathtaking view of the ocean. After participants were presented with each of these primes, they were tested for subtle changes in attitudes to religious belief by rating agreement with the statements ‘God or some supreme being exists’ and ‘God is an active agent in the world’. In healthy volunteers and right- but not left-onset patients, religious belief-scores significantly increased following the aesthetic prime consisting of the ocean view (a wonderful reward) but not the death prime. (The religious ritual prime increased religious belief only inconsistently, with little impact compared with that of the ocean view.) The results directly refuted the anxiety theory of religion while supporting the notion that religiosity was spurred by the quest for unexpected reward. What does all this say about how religion can produce both extraordinarily life-giving and generative human beings (holy men and women) and extraordinary monsters? The same mechanism that enhances our creativity – juicing up the right-sided limbic and prefrontal brain regions with dopamine – also opens us up to religious ideas and experience. But if these brain circuits are pushed too far, thinking becomes not merely divergent but outright deviant and psychotic. Since at least the upper Paleolithic, religious cultures have been shaping, channelling and nourishing the quest for unexpected rewards. Today, such cultures as science, art, music, literature and philosophy confer the same sense of transcendence that religion has done in the past. The trick is tapping the god effect, inducing that altered state of wonder and making the breakthrough to greatness without going over the edge.
"""
essay_for_part3_2 = """Albert Einstein’s theory of general relativity is a century old next year and, as far as the test of time is concerned, it seems to have done rather well. For many, indeed, it doesn’t merely hold up: it is the archetype for what a scientific theory should look like. Einstein’s achievement was to explain gravity as a geometric phenomenon: a force that results from the distortion of space-time by matter and energy, compelling objects – and light itself – to move along particular paths, very much as rivers are constrained by the topography of their landscape. General relativity departs from classical Newtonian mechanics and from ordinary intuition alike, but its predictions have been verified countless times. In short, it is the business. Einstein himself seemed rather indifferent to the experimental tests, however. The first came in 1919, when the British physicist Arthur Eddington observed the Sun’s gravity bending starlight during a solar eclipse. What if those results hadn’t agreed with the theory? (Some accuse Eddington of cherry-picking the figures anyway, but that’s another story.) ‘Then,’ said Einstein, ‘I would have been sorry for the dear Lord, for the theory is correct.’ That was Einstein all over. As the Danish physicist Niels Bohr commented at the time, he was a little too fond of telling God what to do. But this wasn’t sheer arrogance, nor parental pride in his theory. The reason Einstein felt general relativity must be right is that it was too beautiful a theory to be wrong. This sort of talk both delights today’s physicists and makes them a little nervous. After all, isn’t experiment – nature itself – supposed to determine truth in science? What does beauty have to do with it? ‘Aesthetic judgments do not arbitrate scientific discourse,’ the string theorist Brian Greene reassures his readers in The Elegant Universe (1999), the most prominent work of physics exposition in recent years. ‘Ultimately, theories are judged by how they fare when faced with cold, hard, experimental facts.’ Einstein, Greene insists, didn’t mean to imply otherwise – he was just saying that beauty in a theory is a good guide, an indication that you are on the right track. Einstein isn’t around to argue, of course, but I think he would have done. It was Einstein, after all, who said that ‘the only physical theories that we are willing to accept are the beautiful ones’. And if he was simply defending theory against too hasty a deference to experiment, there would be plenty of reason to side with him – for who is to say that, in case of a discrepancy, it must be the theory and not the measurement that is in error? But that’s not really his point. Einstein seems to be asserting that beauty trumps experience come what may. He wasn’t alone. Here’s the great German mathematician Hermann Weyl, who fled Nazi Germany to become a colleague of Einstein’s at the Institute of Advanced Studies in Princeton: ‘My work always tries to unite the true with the beautiful; but when I had to choose one or the other, I usually chose the beautiful.’ So much for John Keats’s ‘Beauty is truth, truth beauty.’ And so much, you might be tempted to conclude, for scientists’ devotion to truth: here were some of its greatest luminaries, pledging obedience to a different calling altogether. Was this kind of talk perhaps just the spirit of the age, a product of fin de siècle romanticism? It would be nice to think so. In fact, the discourse about aesthetics in scientific ideas has never gone away. Even Lev Landau and Evgeny Lifshitz, in their seminal but pitilessly austere midcentury Course of Theoretical Physics, were prepared to call general relativity ‘probably the most beautiful of all existing theories’. Today, popularisers such as Greene are keen to make beauty a selling point of physics. Writing in this magazine last year, the quantum theorist Adrian Kent speculated that the very ugliness of certain modifications of quantum mechanics might count against their credibility. After all, he wrote, here was a field in which ‘elegance seems to be a surprisingly strong indicator of physical relevance’. We have to ask: what is this beauty they keep talking about? Some scientists are a little coy about that. The Nobel Prize-winning physicist Paul Dirac agreed with Einstein, saying in 1963 that ‘it is more important to have beauty in one’s equations than to have them fit experiment’ (how might Greene explain that away?). Yet faced with the question of what this all-important beauty is, Dirac threw up his hands. Mathematical beauty, he said, ‘cannot be defined any more than beauty in art can be defined’ – though he added that it was something ‘people who study mathematics usually have no difficulty in appreciating’. That sounds rather close to the ‘good taste’ of his contemporaneous art critics; we might fear that it amounts to the same mixture of prejudice and paternalism. Given this history of evasion, it was refreshing last November to hear the theoretical physicist Nima Arkani-Hamed spell out what ‘beauty’ really means for him and his colleagues. He was talking to the novelist Ian McEwan at the Science Museum in London, during the opening of the museum’s exhibition on the Large Hadron Collider. ‘Ideas that we find beautiful,’ Arkani-Hamed explained, ‘are not a capricious aesthetic judgment’: It’s not fashion, it’s not sociology. It’s not something that you might find beautiful today but won’t find beautiful 10 years from now. The things that we find beautiful today we suspect would be beautiful for all eternity. And the reason is, what we mean by beauty is really a shorthand for something else. The laws that we find describe nature somehow have a sense of inevitability about them. There are very few principles and there’s no possible other way they could work once you understand them deeply enough. So that’s what we mean when we say ideas are beautiful. Does this bear any relation to what beauty means in the arts? Arkani-Hamed had a shot at that. Take Ludwig van Beethoven, he said, who strove to develop his Fifth Symphony in ‘perfect accordance to its internal logical structure’. it is precisely this that delights mathematicians in a great proof: not that it is correct but that it shows a tangibly human genius Beethoven is indeed renowned for the way he tried out endless variations and directions in his music, turning his manuscripts into inky thickets in his search for the ‘right’ path. Novelists and poets, too, can be obsessive in their pursuit of the mot juste. Reading the novels of Patrick White or the late works of Penelope Fitzgerald, you get the same feeling of almost logical necessity, word by perfect word. But you notice this quality precisely because it is so rare. What generally brings a work of art alive is not its inevitability so much as the decisions that the artist made. We gasp not because the words, the notes, the brushstrokes are ‘right’, but because they are revelatory: they show us not a deterministic process but a sensitive mind making surprising and delightful choices. In fact, pure mathematicians often say that it is precisely this quality that delights them in a great proof: not that it is correct but that it shows a personal, tangibly human genius taking steps in a direction we’d never have guessed. ‘The things that we find beautiful today we suspect would be beautiful for all eternity’: here is where Arkani-Hamed really scuppers the notion that the kind of beauty sought by science has anything to do with the major currents of artistic culture. After all, if there’s one thing you can say about beauty, it is that the beholder has a lot to do with it. We can still find beauty in the Paleolithic paintings at Lascaux and the music of William Byrd, while admitting that a heck of a lot of beauty really is fashion and sociology. Why shouldn’t it be? How couldn’t it be? We still swoon at Jan van Eyck. Would van Eyck’s audience swoon at Mark Rothko? The gravest offenders in this attempted redefinition of beauty are, of course, the physicists. This is partly because their field has always been heir to Platonism – the mystical conviction of an orderly cosmos. Such a belief is almost a precondition for doing physics in the first place: what’s the point in looking for rules unless you believe they exist? The MIT physicist Max Tegmark now goes so far as to say that mathematics constitutes the basic fabric of reality, a claim redolent of Plato’s most extreme assertions in Timaeus. But Platonism will not connect you with the mainstream of aesthetic thought – not least because Plato himself was so distrustful of art (he banned the lying poets from his Republic, after all). Better that we turn to Immanuel Kant. Kant expended considerable energies in his Critique of Judgment (1790) trying to disentangle the aesthetic aspects of beauty from the satisfaction one feels in grasping an idea or recognizing a form, and it does us little good to jumble them up again. All that conceptual understanding gives us, he concluded, is ‘the solution that satisfies the problem… not a free and indeterminately final entertainment of the mental powers with what is called beautiful’. Beauty, in other words, is not a resolution: it opens the imagination. Physicists might be the furthest gone along Plato’s trail, but they are not alone. Consider the many chemists whose idea of beauty seems to be dictated primarily by the molecules they find pleasing – usually because of some inherent mathematical symmetry, such as in the football-shaped carbon molecule buckminsterfullerene (strictly speaking, a truncated icosahedron). Of course, this is just another instance of mathematics-worship, yoking beauty to qualities of regularity that were not deemed artistically beautiful even in antiquity. Brian Greene claims: ‘In physics, as in art, symmetry is a key part of aesthetics.’ Yet for Plato it was precisely art’s lack of symmetry (and thus intelligibility) that denied it access to real beauty. Art was just too messy to be beautiful. In seeing matters the other way around, Kant speaks for the mainstream of artistic aesthetics: ‘All stiff regularity (such as approximates to mathematical regularity) has something in it repugnant to taste.’ We weary of it, as we do a nursery rhyme. Or as the art historian Ernst Gombrich put it in 1988, too much symmetry ensures that ‘once we have grasped the principle of order… it holds no more surprise’. Artistic beauty, Gombrich believed, relies on a tension between symmetry and asymmetry: ‘a struggle between two opponents of equal power, the formless chaos, on which we impose our ideas, and the all-too-formed monotony, which we brighten up by new accents’. Even Francis Bacon (the 17th-century proto-scientist, not the 20th-century artist) understood this much: ‘There is no excellent beauty that hath not some strangeness in the proportion.’ Perhaps I have been a little harsh on the chemists – those cube- and prism-shaped molecules are fun in their own way. But Bacon, Kant and Gombrich are surely right to question their aesthetic merit. As the philosopher of chemistry Joachim Schummer pointed out in 2003, it is simply parochial to redefine beauty as symmetry: doing so cuts one off from the dominant tradition in artistic theory. There’s a reason why our galleries are not, on the whole, filled with paintings of perfect spheres. Why shouldn’t scientists be allowed their own definition of beauty? Perhaps they should. Yet isn’t there a narrowness to the standard that they have chosen? Even that might not be so bad, if their cult of ‘beauty’ didn’t seem to undermine the credibility of what they otherwise so strenuously assert: the sanctity of evidence. It doesn’t matter who you are, they say, how famous or erudite or well-published: if your theory doesn’t match up to nature, it’s history. But if that’s the name of the game, why on earth should some vague notion of beauty be brought into play as an additional arbiter? Because of experience, they might reply: true theories are beautiful. Well, general relativity might have turned out OK, but plenty of others have not. Take the four-colour theorem: the proposal that it is possible to colour any arbitrary patchwork in just four colours without any patches of the same colour touching one another. In 1879 it seemed as though the British mathematician Alfred Kempe had found a proof – and it was widely accepted for a decade, because it was thought beautiful. It was wrong. The current proof is ugly as heck – it relies on a brute-force exhaustive computer search, which some mathematicians refuse to accept as a valid form of demonstration – but it might turn out to be all there is. The same goes for Andrew Wiles’s proof of Fermat’s Last Theorem, first announced in 1993. The basic theorem is wonderfully simple and elegant, the proof anything but: 100 pages long and more complex than the Pompidou Centre. There’s no sign of anything simpler. It’s not hard to mine science history for theories and proofs that were beautiful and wrong, or complicated and right. No one has ever shown a correlation between beauty and ‘truth’. But it is worse than that, for sometimes ‘beauty’ in the sense that many scientists prefer – an elegant simplicity, to put it in crude terms – can act as a fake trump card that deflects inquiry. In one little corner of science that I can claim to know reasonably well, an explanation from 1959 for why water-repelling particles attract when immersed in water (that it’s an effect of entropy, there being more disordered water molecules when the particles stick together) was so neat and satisfying that it continues to be peddled today, even though the experimental data show that it is untenable and that the real explanation probably lies in a lot of devilish detail. I would be thrilled if the artist were to say to the scientist: ‘No, we’re not even on the same page’ Might it even be that the marvellous simplicity and power of natural selection strikes some biologists as so beautiful an idea – an island of order in a field otherwise beset with caveats and contradictions – that it must be defended at any cost? Why else would attempts to expose its limitations, exceptions and compromises still ignite disputes pursued with near-religious fervour? The idea that simplicity, as distinct from beauty, is a guide to truth – the idea, in other words, that Occam’s Razor is a useful tool – seems like something of a shibboleth in itself. As these examples show, it is not reliably correct. Perhaps it is a logical assumption, all else being equal. But it is rare in science that all else is equal. More often, some experiments support one theory and others another, with no yardstick of parsimony to act as referee. We can be sure, however, that simplicity is not the ultimate desideratum of aesthetic merit. Indeed, in music and visual art, there appears to be an optimal level of complexity below which preference declines. A graph of enjoyment versus complexity has the shape of an inverted U: there is a general preference for, say, ‘Eleanor Rigby’ over both ‘Baa Baa Black Sheep’ and Pierre Boulez’s Structures Ia, just as there is for lush landscapes over monochromes. For most of us, our tastes eschew the extremes. Ironically, the quest for a ‘final theory’ of nature’s deepest physical laws has meant that the inevitability and simplicity that Arkani-Hamed prizes so highly now look more remote than ever. For we are now forced to contemplate no fewer than 10500 permissible variants of string theory. It’s always possible that 10500 minus one of them might vanish at a stroke, thanks to the insight of some future genius. Right now, though, the dream of elegant fundamental laws lies in bewildering disarray. An insistence that the ‘beautiful’ must be true all too easily elides into an empty circularity: what is true must therefore be beautiful. I see this in the conviction of many chemists that the periodic table, with all its backtracking sequences of electron shells, its positional ambiguities for elements such as hydrogen and unsightly bulges that the flat page can’t constrain, is a thing of loveliness. There, surely, speaks the voice of duty, not genuine feeling. The search for an ideal, perfect Platonic form of the table amid spirals, hypercubes and pyramids has an air of desperation. Despite all this, I don’t want scientists to abandon their talk of beauty. Anything that inspires scientific thinking is valuable, and if a quest for beauty – a notion of beauty peculiar to science, removed from art – does that, then bring it on. And if it gives them a language in which to converse with artists, rather than standing on soapboxes and trading magisterial insults like C P Snow and F R Leavis, all the better. I just wish they could be a bit more upfront about the fact that they are (as is their wont) torturing a poor, fuzzy, everyday word to make it fit their own requirements. I would be rather thrilled if the artist, rather than accepting this unified pursuit of beauty (as Ian McEwan did), were to say instead: ‘No, we’re not even on the same page. This beauty of yours means nothing to me.’ If, on the other hand, we want beauty in science to make contact with aesthetics in art, I believe we should seek it precisely in the human aspect: in ingenious experimental design, elegance of theoretical logic, gentle clarity of exposition, imaginative leaps of reasoning. These things are not vital for a theory that works, an experiment that succeeds, an explanation that enchants and enlightens. But they are rather lovely. Beauty, unlike truth or nature, is something we make ourselves.
"""

essays = [essay_for_part3_1, essay_for_part3_2]



In [ ]:
#Generate Summaries
print("\n--- LLM Prompting Results ---")

for i, essay_text in enumerate(essays):
    print(f"\n" + "="*30)
    print(f"        ESSAY {i+1}")
    print("="*30)
    print(f"**Original Essay (snippet):**\n{essay_text[:500]}...\n")

    for j, prompt_template in enumerate(prompts):
        print(f"\n--- Prompt {j+1} ---")
        full_prompt = prompt_template.format(essay_text=essay_text)


        model_inputs = llm_tokenizer(full_prompt, return_tensors="pt").to(device)


        generated_ids = llm_model.generate(
            **model_inputs,
            max_new_tokens=200,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7
        )

        prompt_token_length = model_inputs.input_ids.shape[1]
        decoded_summary = llm_tokenizer.decode(
            generated_ids[0][prompt_token_length:],
            skip_special_tokens=True
        )

        print(f"**Generated Summary:**\n{decoded_summary}\n")

# TODO: Add your write-up comments here, qualitatively comparing
# the results from Prompt 1 vs. Prompt 2 and how they
# compare to the T5 models and the ground truth.

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



--- LLM Prompting Results ---

        ESSAY 1
**Original Essay (snippet):**
When I was 12, on family vacation in New Mexico, I watched a group of elaborately-costumed Navajo men belt out one intimidating song after the next. They executed a set of beautifully coordinated dance turns to honour the four cardinal directions, each one symbolising sacred gifts from the gods. Yet the tourist-packed audience lost interest and my family, too, prepared to leave. Then, all of a sudden, the dancers were surprised by a haunting, muscled old man adorned with strange pendants, animal...


--- Prompt 1 ---


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


**Generated Summary:**
In this essay, the author reflects on a childhood experience of witnessing a Navajo dance performance and the encounter with a holy man, or shaman, which left a lasting impact. Throughout her neuroscience research, she discovered that the right arrangement of neural circuitry and chemistry, specifically the neurotransmitter dopamine, can lead to extraordinary and holy individuals or delusional and violent fanatics. The author explores the origins of religious behaviors, dating back to the upper Paleolithic period when a 12-year-old boy made a handprint in a cave as a religious offering. She suggests that the ability to entertain unusual ideas and associations, driven by a balance of dopamine in the brain's limbic and prefrontal regions, can result in creativity, inspired leadership, and profound religious experience, but also fanaticism and mental illness in genetically vulnerable individuals. The author's research on Parkinson's Disease patients led


--- Prompt

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


**Generated Summary:**
This essay recounts a formative experience at age 12 when the author witnessed a Navajo dance performance and encountered a terrifying shaman. The encounter sparked a lifelong interest in understanding the neural basis for extraordinary individuals, from religious leaders to creative geniuses. The essay discusses how dopamine, a neurotransmitter linked to emotion and motivation, plays a crucial role in shaping religious experiences and creative thinking. The author hypothesizes that an optimal level of dopamine activity can lead to heightened creativity and spiritual experiences, while excess dopamine can result in fanaticism and violence. The essay also explores the relationship between dopamine and religious experiences, drawing on historical and archaeological evidence.


        ESSAY 2
**Original Essay (snippet):**
Albert Einstein’s theory of general relativity is a century old next year and, as far as the test of time is concerned, it seems to have done rat

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


**Generated Summary:**
Albert Einstein's theory of general relativity, which explains gravity as a geometric phenomenon, has held up well for a century and is often considered the archetype of a scientific theory. Einstein himself was deeply invested in the theory's beauty, viewing it as a guide to the right track in science despite the importance of experimental facts. The essay explores the concept of beauty in science, with some scientists arguing that it is a useful indicator of physical relevance and others questioning its relationship to truth and experiment. Theoretical physicist Nima Arkani-Hamed defines scientific beauty as ideas that have a sense of inevitability and few principles, while acknowledging its differences from artistic beauty. The essay also discusses the history of aesthetic judgments in science and argues that scientists should be upfront about their use of the term "beauty" to describe their theories.


--- Prompt 2 ---
**Generated Summary:**
This essay explor